# Huấn luyện Tổng hợp 3 Phương pháp Đối chứng: Paper 1 (modelmethod1.pt), Paper 4 (modelmethod4.pt) và Paper 9 (modelmethod9.pt)


In [ ]:
import os
import re
import gc
import sys
import math
import json
import time
import shutil
import random
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Optional, Any

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM
from transformers.modeling_outputs import BaseModelOutput
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score, mean_absolute_error

import nltk
for pkg in ["stopwords", "punkt", "wordnet", "omw-1.4", "averaged_perceptron_tagger"]:
    try:
        nltk.data.find(f"corpora/{pkg}" if pkg in ["stopwords", "wordnet", "omw-1.4"] else (f"tokenizers/{pkg}" if pkg == "punkt" else f"taggers/{pkg}"))
    except LookupError:
        try:
            nltk.download(pkg, quiet=True)
        except Exception:
            pass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"PyTorch: {torch.__version__} | Device: {DEVICE} | GPUs: {NUM_GPUS}")


## 1. Thiết lập Cấu hình Chung và Nạp Tập dữ liệu hotel_reviews.parquet


In [ ]:
class UnifiedBenchmarkConfig:
    PARQUET_CANDIDATES = [
        "/kaggle/input/dap_dts/hotel_reviews.parquet",
        "/kaggle/input/dap-dts/hotel_reviews.parquet",
        "/kaggle/input/hotel-reviews-cram-absa/hotel_reviews.parquet",
        "/kaggle/input/tripadvisor-hotel-reviews/hotel_reviews.parquet",
        "data/df/hotel_reviews.parquet",
        "hotel_reviews.parquet"
    ]
    OUTPUT_DIR = Path("/kaggle/working" if Path("/kaggle/working").exists() else "model")
    LOCAL_MODEL_DIR = Path("model")

    MODEL1_SAVE_PATH = OUTPUT_DIR / "modelmethod1.pt"
    LOCAL_MODEL1_SAVE_PATH = LOCAL_MODEL_DIR / "modelmethod1.pt"
    MODEL4_SAVE_PATH = OUTPUT_DIR / "modelmethod4.pt"
    LOCAL_MODEL4_SAVE_PATH = LOCAL_MODEL_DIR / "modelmethod4.pt"
    MODEL9_SAVE_PATH = OUTPUT_DIR / "modelmethod9.pt"
    LOCAL_MODEL9_SAVE_PATH = LOCAL_MODEL_DIR / "modelmethod9.pt"
    SUMMARY_SAVE_PATH = OUTPUT_DIR / "history_all_methods.json"

    BACKBONE_NAME = "bert-base-uncased"
    GEN_MODEL_NAME = "google/flan-t5-small"
    CANONICAL_ASPECTS = ["Service", "Facility", "Experience", "Amenity", "Loyalty", "Branding"]
    NUM_ASPECTS = 6
    NUM_SENTIMENT_CLASSES = 4
    NUM_STARS = 5
    NUM_SOFT_PROMPT_TOKENS = 4
    MAX_SEQ_LEN = 256
    MAX_COMMENT_LEN = 48
    NUM_WORKERS = 2 if os.name != "nt" else 0
    PIN_MEMORY = torch.cuda.is_available()

    P1_HIDDEN_BERT = 768
    P1_HIDDEN_CLS = 50
    P1_BATCH_SIZE = 16 * max(1, NUM_GPUS)
    P1_EPOCHS = 2
    P1_LR = 2e-5

    P4_W2V_DIM = 100
    P4_W2V_WINDOW = 5
    P4_W2V_MIN_COUNT = 5
    P4_BM25_B = 0.8
    P4_BM25_K1 = 2.0
    P4_TFIGM_LAMBDA = 7.0
    P4_MAX_VOCAB = 6000
    P4_CNN_REGIONS = (2, 3, 4)
    P4_CNN_FILTERS = 2
    P4_MAX_LEN = 128
    P4_BATCH_SIZE = 64
    P4_EPOCHS = 8
    P4_LR = 1e-3

    P9_BERT_DIM = 768
    P9_BILSTM_UNITS = 128
    P9_GCN_LAYERS = 2
    P9_GCN_DIM = 128
    P9_DROPOUT = 0.5
    P9_BATCH_SIZE = 32 * max(1, NUM_GPUS)
    P9_EPOCHS = 6
    P9_LR_BERT = 3e-5
    P9_LR_HEADS = 1e-3

cfg = UnifiedBenchmarkConfig()
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cfg.LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

ASPECT_ID_MAP = {asp: idx + 1 for idx, asp in enumerate(cfg.CANONICAL_ASPECTS)}
ASPECT_ID_MAP["None"] = 0
SENTIMENT_TO_ID = {"Negative": 1, "Neutral": 2, "Positive": 3}
ID_TO_SENTIMENT = {0: "Not Mentioned", 1: "Negative", 2: "Neutral", 3: "Positive"}

HEDGING_MODIFIERS = {
    "moderate", "okay", "ok", "decent", "fair", "acceptable", "average", "slightly",
    "bit", "ordinary", "normal", "passable", "tolerable", "adequate", "fine", "basic"
}

try:
    from nltk.corpus import stopwords
    NLTK_STOPWORDS = set(stopwords.words("english"))
except Exception:
    NLTK_STOPWORDS = {
        "a", "an", "the", "and", "or", "if", "in", "on", "at", "to", "from", "by", "for",
        "with", "about", "as", "into", "like", "through", "after", "over", "between", "out",
        "of", "during", "before", "under", "around", "among", "is", "are", "was", "were",
        "be", "been", "being", "have", "has", "had", "do", "does", "did", "will", "would",
        "shall", "should", "can", "could", "may", "might", "must", "i", "me", "my", "myself",
        "we", "our", "ours", "ourselves", "you", "your", "yours", "he", "him", "his", "she",
        "her", "hers", "it", "its", "they", "them", "their", "theirs", "this", "that", "these",
        "those", "am", "so", "than", "too", "very", "just", "where", "when", "who", "whom",
        "which", "what", "there", "here", "all", "any", "both", "each", "few", "more", "most",
        "other", "some", "such", "only", "own", "same", "then", "once", "not", "no"
    }

def parse_annotations_safe(anns: Any) -> List[Dict[str, Any]]:
    if anns is None:
        return []
    if isinstance(anns, np.ndarray):
        anns = anns.tolist()
    elif isinstance(anns, str):
        try:
            anns = json.loads(anns)
        except Exception:
            return []
    if not isinstance(anns, list):
        return []
    return [dict(a) for a in anns if isinstance(a, dict)]

parquet_path = next((Path(p) for p in cfg.PARQUET_CANDIDATES if Path(p).exists()), None)
if parquet_path is None and Path("/kaggle/input").exists():
    found = list(Path("/kaggle/input").rglob("hotel_reviews.parquet"))
    if found:
        parquet_path = found[0]
if parquet_path is None:
    raise FileNotFoundError("Không tìm thấy hotel_reviews.parquet!")

df_master = pd.read_parquet(parquet_path)
if "ordinal_rating" not in df_master.columns:
    df_master["ordinal_rating"] = df_master["rating_5"].clip(1, 5).astype(int) - 1

df_train_raw = df_master[df_master["split"] == "train"].reset_index(drop=True)
df_val = df_master[df_master["split"] == "val"].reset_index(drop=True)
df_test = df_master[df_master["split"] == "test"].reset_index(drop=True)

bert_tokenizer = AutoTokenizer.from_pretrained(cfg.BACKBONE_NAME)
gen_tokenizer = AutoTokenizer.from_pretrained(cfg.GEN_MODEL_NAME)
all_methods_summary = {}
print(f"Đã nạp dữ liệu: Train={len(df_train_raw):,} | Val={len(df_val):,} | Test={len(df_test):,}")


## 2. Phương pháp 1 — Paper 1: BERT-ABSA + Random Oversampling (Bimaputra & Sutoyo, 2023) -> modelmethod1.pt


In [ ]:
CONTRACTIONS_PAPER1 = {
    "can't": "cannot", "won't": "will not", "don't": "do not", "doesn't": "does not",
    "didn't": "did not", "isn't": "is not", "aren't": "are not", "wasn't": "was not",
    "weren't": "were not", "haven't": "have not", "hasn't": "has not", "hadn't": "had not",
    "couldn't": "could not", "wouldn't": "would not", "shouldn't": "should not",
    "n't": " not", "'re": " are", "'ve": " have", "'ll": " will", "'m": " am"
}

def paper1_clean_text(text: str) -> str:
    txt = unicodedata.normalize("NFC", str(text)).lower()
    for k, v in CONTRACTIONS_PAPER1.items():
        txt = re.sub(r"\b" + re.escape(k) + r"\b", v, txt)
        txt = txt.replace(k, v)
    txt = re.sub(r"@\w+|https?://\S+|www\.\S+", " ", txt)
    tokens = re.findall(r"[a-z]+", txt)
    filtered = [t for t in tokens if t not in NLTK_STOPWORDS and len(t) > 1]
    return " ".join(filtered) if filtered else txt

def apply_random_oversampling(df_tr: pd.DataFrame) -> pd.DataFrame:
    counts = df_tr["ordinal_rating"].value_counts()
    max_cnt = int(counts.max())
    target_per_minority = max(int(max_cnt * 0.35), int(counts.median()))
    frames = []
    rng = np.random.default_rng(SEED)
    for star_cls, grp in df_tr.groupby("ordinal_rating"):
        if len(grp) < target_per_minority:
            extra_idx = rng.choice(grp.index.values, size=target_per_minority - len(grp), replace=True)
            frames.append(pd.concat([grp, df_tr.loc[extra_idx]], ignore_index=True))
        else:
            frames.append(grp)
    df_os = pd.concat(frames, ignore_index=True)
    return df_os.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

class Paper1ABSADataset(Dataset):
    def __init__(self, df: pd.DataFrame, bert_tok, seq2seq_tok, max_len: int = 256, max_comment_len: int = 48):
        self.df = df.reset_index(drop=True)
        self.bert_tok = bert_tok
        self.seq2seq_tok = seq2seq_tok
        self.max_len = max_len
        self.max_comment_len = max_comment_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        raw_text = unicodedata.normalize("NFC", str(row["review_text"]))
        cleaned_text = paper1_clean_text(raw_text)

        enc = self.bert_tok(
            cleaned_text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        tok_strs = self.bert_tok.convert_ids_to_tokens(input_ids.tolist())

        ordinal_rating = int(row["ordinal_rating"])
        aspect_sent_labels = torch.zeros(cfg.NUM_ASPECTS, dtype=torch.long)
        aspect_star_labels = torch.full((cfg.NUM_ASPECTS,), ordinal_rating, dtype=torch.long)
        rationale_masks = torch.zeros((cfg.NUM_ASPECTS, self.max_len), dtype=torch.float32)
        gen_span_ids = torch.zeros((cfg.NUM_ASPECTS, self.max_comment_len), dtype=torch.long)
        gen_span_mask = torch.zeros((cfg.NUM_ASPECTS, self.max_comment_len), dtype=torch.long)

        anns = parse_annotations_safe(row.get("aspect_annotations", []))
        votes = defaultdict(list)
        span_toks_map = defaultdict(set)
        span_texts_map = defaultdict(list)
        for ann in anns:
            asp = ann.get("aspect")
            sent = ann.get("sentiment", "Neutral")
            if asp in ASPECT_ID_MAP and asp != "None":
                a_idx = ASPECT_ID_MAP[asp] - 1
                votes[a_idx].append(SENTIMENT_TO_ID.get(sent, 2))
                raw_sp = str(ann.get("span_text", "")).strip()
                if raw_sp:
                    span_texts_map[a_idx].append(raw_sp)
                span_toks_map[a_idx].update(paper1_clean_text(raw_sp).split())

        for a_idx, v_list in votes.items():
            maj = Counter(v_list).most_common(1)[0][0]
            aspect_sent_labels[a_idx] = maj
            joined_sp = " ".join(span_texts_map[a_idx]).strip()
            sp_words = re.findall(r"[a-z]+", (joined_sp if joined_sp else raw_text).lower())
            has_hedge = any(w in HEDGING_MODIFIERS for w in sp_words)
            if maj == 3:
                aspect_star_labels[a_idx] = 3 if has_hedge else max(3, ordinal_rating)
            elif maj == 2:
                aspect_star_labels[a_idx] = 2
            else:
                aspect_star_labels[a_idx] = 1 if has_hedge else min(1, ordinal_rating)

            target_words = span_toks_map[a_idx]
            if target_words:
                for t_i, t_str in enumerate(tok_strs):
                    clean_sub = t_str.replace("##", "")
                    if clean_sub in target_words and attention_mask[t_i].item() == 1:
                        rationale_masks[a_idx, t_i] = 1.0

            if joined_sp:
                t_enc = self.seq2seq_tok(
                    joined_sp,
                    max_length=self.max_comment_len,
                    padding="max_length",
                    truncation=True,
                    return_tensors="pt"
                )
                gen_span_ids[a_idx] = t_enc["input_ids"].squeeze(0)
                gen_span_mask[a_idx] = t_enc["attention_mask"].squeeze(0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "aspect_sent_labels": aspect_sent_labels,
            "aspect_star_labels": aspect_star_labels,
            "rationale_masks": rationale_masks,
            "gen_span_ids": gen_span_ids,
            "gen_span_mask": gen_span_mask,
            "ordinal_rating": torch.tensor(ordinal_rating, dtype=torch.long)
        }

class Paper1BERTModel(nn.Module):
    def __init__(self, cfg: UnifiedBenchmarkConfig):
        super().__init__()
        self.num_aspects = cfg.NUM_ASPECTS
        self.max_comment_len = cfg.MAX_COMMENT_LEN
        self.num_prompt_tokens = cfg.NUM_SOFT_PROMPT_TOKENS

        self.bert = AutoModel.from_pretrained(cfg.BACKBONE_NAME)
        self.gen_model = AutoModelForSeq2SeqLM.from_pretrained(cfg.GEN_MODEL_NAME)
        self.gen_d_model = self.gen_model.config.d_model
        for param in self.gen_model.parameters():
            param.requires_grad = False

        self.aspect_hidden_layers = nn.ModuleList([
            nn.Sequential(nn.Linear(cfg.P1_HIDDEN_BERT, cfg.P1_HIDDEN_CLS), nn.ReLU(), nn.Dropout(0.1))
            for _ in range(cfg.NUM_ASPECTS)
        ])
        self.aspect_heads = nn.ModuleList([
            nn.Linear(cfg.P1_HIDDEN_CLS, cfg.NUM_SENTIMENT_CLASSES)
            for _ in range(cfg.NUM_ASPECTS)
        ])
        self.aspect_star_heads = nn.ModuleList([
            nn.Sequential(nn.Linear(cfg.P1_HIDDEN_BERT, cfg.P1_HIDDEN_CLS), nn.ReLU(), nn.Dropout(0.1), nn.Linear(cfg.P1_HIDDEN_CLS, cfg.NUM_STARS))
            for _ in range(cfg.NUM_ASPECTS)
        ])
        self.review_star_head = nn.Sequential(
            nn.Linear(cfg.P1_HIDDEN_BERT, cfg.P1_HIDDEN_CLS), nn.ReLU(), nn.Dropout(0.1), nn.Linear(cfg.P1_HIDDEN_CLS, cfg.NUM_STARS)
        )
        self.token_scorer = nn.Linear(cfg.P1_HIDDEN_BERT, cfg.NUM_ASPECTS)
        self.aspect_soft_prompt = nn.Sequential(
            nn.Linear(cfg.P1_HIDDEN_BERT + cfg.P1_HIDDEN_CLS, 256),
            nn.GELU(),
            nn.Linear(256, self.num_prompt_tokens * self.gen_d_model),
            nn.LayerNorm(self.num_prompt_tokens * self.gen_d_model)
        )

    def forward(self, input_ids, attention_mask, gen_span_ids=None, gen_span_mask=None):
        bsz = input_ids.size(0)
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        seq_h = out.last_hidden_state
        cls_h = seq_h[:, 0, :]

        tok_scores = torch.sigmoid(self.token_scorer(seq_h)).transpose(1, 2) * attention_mask.unsqueeze(1).float()
        denom = tok_scores.sum(dim=-1, keepdim=True).clamp(min=1e-4)
        rat_pooled = torch.einsum("bal,bld->bad", tok_scores, seq_h) / denom

        asp_logits, asp_star_logits, soft_prompts = [], [], []
        for a_idx in range(self.num_aspects):
            h_50 = self.aspect_hidden_layers[a_idx](cls_h)
            asp_logits.append(self.aspect_heads[a_idx](h_50))
            asp_star_logits.append(self.aspect_star_heads[a_idx](cls_h))
            sp_in = torch.cat([rat_pooled[:, a_idx, :], h_50], dim=-1)
            soft_prompts.append(self.aspect_soft_prompt(sp_in).view(bsz, self.num_prompt_tokens, self.gen_d_model))

        asp_logits = torch.stack(asp_logits, dim=1)
        asp_star_logits = torch.stack(asp_star_logits, dim=1)
        soft_prompts = torch.stack(soft_prompts, dim=1)
        rev_star_logits = self.review_star_head(cls_h)

        gen_loss = torch.zeros(1, device=input_ids.device, dtype=torch.float32)
        if gen_span_ids is not None and gen_span_mask is not None:
            flat_prompts = soft_prompts.view(bsz * self.num_aspects, self.num_prompt_tokens, self.gen_d_model)
            flat_ids = gen_span_ids.view(bsz * self.num_aspects, -1)
            flat_mask = gen_span_mask.view(bsz * self.num_aspects, -1)
            valid_rows = flat_mask.sum(dim=-1) > 1
            if valid_rows.any():
                active_prompts = flat_prompts[valid_rows].float()
                active_ids = flat_ids[valid_rows]
                active_mask = flat_mask[valid_rows]
                with torch.no_grad():
                    t5_tok_emb = self.gen_model.get_input_embeddings()(active_ids).float()
                    mask_f = active_mask.unsqueeze(-1).float()
                    target_vec = (t5_tok_emb * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)
                pred_vec = active_prompts.mean(dim=1)
                gen_loss = ((1.0 - F.cosine_similarity(pred_vec, target_vec, dim=-1).mean()) + F.smooth_l1_loss(pred_vec, target_vec)).view(1)

        return {
            "aspect_logits": asp_logits,
            "aspect_star_logits": asp_star_logits,
            "rev_star_logits": rev_star_logits,
            "rationale_probs": tok_scores,
            "soft_prompts": soft_prompts,
            "gen_loss": gen_loss
        }

    @torch.no_grad()
    def generate_aspect_commentary(self, aspect_idx, aspect_name, sentiment_label, star_val, review_text, outputs, input_ids, attention_mask, bert_tok, seq2seq_tok):
        device = input_ids.device
        rat_prob = outputs["rationale_probs"][0, aspect_idx]
        valid_len = int(attention_mask[0].sum().item())
        tok_ids = input_ids[0, :valid_len].cpu().tolist()
        scores = rat_prob[:valid_len].cpu().numpy()

        sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", review_text.strip()) if s.strip()]
        if len(sentences) > 1:
            scored_sents = []
            for s in sentences:
                s_ids = set(bert_tok.encode(paper1_clean_text(s), add_special_tokens=False))
                hit_scores = [scores[i] for i in range(1, valid_len - 1) if tok_ids[i] in s_ids]
                scored_sents.append((float(np.mean(hit_scores)) if hit_scores else 0.0, s))
            scored_sents.sort(key=lambda x: x[0], reverse=True)
            focused_evidence = " ".join([s for _, s in scored_sents[:2]])
        else:
            focused_evidence = review_text.strip()

        prompt_str = (
            f"Paraphrase and summarize the guest's opinion on the hotel {aspect_name} "
            f"({sentiment_label}, {star_val}/5 stars) into a short third-person analytical comment: {focused_evidence}"
        )
        p_enc = seq2seq_tok(prompt_str, max_length=192, truncation=True, return_tensors="pt").to(device)
        tok_embeds = self.gen_model.get_input_embeddings()(p_enc["input_ids"]).float()
        soft_p = outputs["soft_prompts"][0:1, aspect_idx, :, :].float()
        combined_embeds = torch.cat([0.20 * soft_p, tok_embeds], dim=1)
        prompt_mask = torch.ones((1, soft_p.size(1)), dtype=p_enc["attention_mask"].dtype, device=device)
        combined_mask = torch.cat([prompt_mask, p_enc["attention_mask"]], dim=1)

        enc_out = self.gen_model.encoder(inputs_embeds=combined_embeds, attention_mask=combined_mask)
        gen_ids = self.gen_model.generate(
            encoder_outputs=enc_out,
            attention_mask=combined_mask,
            max_new_tokens=self.max_comment_len,
            min_length=8,
            num_beams=4,
            no_repeat_ngram_size=3,
            repetition_penalty=1.25,
            early_stopping=True
        )
        return seq2seq_tok.decode(gen_ids[0], skip_special_tokens=True).strip()

print("=== Bắt đầu Huấn luyện Phương pháp 1 (Paper 1: BERT-ABSA + Random Oversampling) ===")
df_train_p1 = apply_random_oversampling(df_train_raw)
p1_train_loader = DataLoader(Paper1ABSADataset(df_train_p1, bert_tokenizer, gen_tokenizer, cfg.MAX_SEQ_LEN, cfg.MAX_COMMENT_LEN), batch_size=cfg.P1_BATCH_SIZE, shuffle=True, num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY)
p1_val_loader = DataLoader(Paper1ABSADataset(df_val, bert_tokenizer, gen_tokenizer, cfg.MAX_SEQ_LEN, cfg.MAX_COMMENT_LEN), batch_size=cfg.P1_BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY)

base_p1 = Paper1BERTModel(cfg).to(DEVICE)
model_p1 = nn.DataParallel(base_p1, device_ids=list(range(NUM_GPUS))) if NUM_GPUS > 1 else base_p1
opt_p1 = torch.optim.Adam([p for p in model_p1.parameters() if p.requires_grad], lr=cfg.P1_LR)
ce_loss_fn = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate_bert_baseline(model, loader):
    model.eval()
    total_loss = 0.0
    star_preds, star_trues = [], []
    asp_preds, asp_trues = [], []
    rat_inter, rat_pred_sum, rat_true_sum = 0.0, 0.0, 0.0
    overlap_rates = []

    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        asp_y = batch["aspect_sent_labels"].to(DEVICE)
        asp_star_y = batch["aspect_star_labels"].to(DEVICE)
        rat_mask = batch["rationale_masks"].to(DEVICE)
        gen_ids = batch["gen_span_ids"].to(DEVICE)
        gen_mask = batch["gen_span_mask"].to(DEVICE)
        star_y = batch["ordinal_rating"].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, gen_span_ids=gen_ids, gen_span_mask=gen_mask)
        g_loss = outputs["gen_loss"].mean()
        loss = (
            ce_loss_fn(outputs["aspect_logits"].view(-1, cfg.NUM_SENTIMENT_CLASSES), asp_y.view(-1))
            + ce_loss_fn(outputs["aspect_star_logits"].view(-1, cfg.NUM_STARS), asp_star_y.view(-1))
            + ce_loss_fn(outputs["rev_star_logits"], star_y)
            + 0.25 * F.binary_cross_entropy(outputs["rationale_probs"].clamp(1e-6, 1.0 - 1e-6), rat_mask)
            + 0.35 * g_loss
        )
        total_loss += loss.item()

        star_preds.extend(outputs["rev_star_logits"].argmax(dim=-1).cpu().numpy())
        star_trues.extend(star_y.cpu().numpy())
        asp_preds.append(outputs["aspect_logits"].argmax(dim=-1).cpu().numpy())
        asp_trues.append(asp_y.cpu().numpy())

        pred_mask = (outputs["rationale_probs"] >= 0.5).float()
        rat_inter += (pred_mask * rat_mask).sum().item()
        rat_pred_sum += pred_mask.sum().item()
        rat_true_sum += rat_mask.sum().item()

        active_per_tok = pred_mask.sum(dim=1)
        leaked = (active_per_tok > 1.0).float().sum().item()
        active = (active_per_tok >= 1.0).float().sum().item()
        overlap_rates.append(leaked / max(active, 1.0))

    y_true_s = np.array(star_trues)
    y_pred_s = np.array(star_preds)
    acc_at_0 = accuracy_score(y_true_s, y_pred_s) * 100.0
    per_star_mae = [np.abs(y_true_s[y_true_s == k] - y_pred_s[y_true_s == k]).mean() for k in range(5) if np.any(y_true_s == k)]
    macro_mae = float(np.mean(per_star_mae)) if per_star_mae else 0.0

    Y_true_a = np.vstack(asp_trues)
    Y_pred_a = np.vstack(asp_preds)
    asp_f1s = [f1_score(Y_true_a[:, a], Y_pred_a[:, a], average="macro", zero_division=0) * 100.0 for a in range(cfg.NUM_ASPECTS)]
    macro_asp_f1 = float(np.mean(asp_f1s))
    rationale_f1 = (2.0 * rat_inter / max(rat_pred_sum + rat_true_sum, 1.0)) * 100.0

    return {
        "val_loss": round(total_loss / max(len(loader), 1), 4),
        "acc_at_0": round(acc_at_0, 2),
        "aspect_macro_f1": round(macro_asp_f1, 2),
        "macro_mae": round(macro_mae, 4),
        "rationale_f1": round(rationale_f1, 2),
        "leakage_rate": round(float(np.mean(overlap_rates)) * 100.0, 2),
        "per_aspect_f1": dict(zip(cfg.CANONICAL_ASPECTS, [round(x, 2) for x in asp_f1s]))
    }

best_p1_f1 = -1.0
p1_history = []
for epoch in range(1, cfg.P1_EPOCHS + 1):
    model_p1.train()
    t0 = time.time()
    running_loss = 0.0
    for batch in p1_train_loader:
        opt_p1.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        asp_y = batch["aspect_sent_labels"].to(DEVICE)
        asp_star_y = batch["aspect_star_labels"].to(DEVICE)
        rat_mask = batch["rationale_masks"].to(DEVICE)
        gen_ids = batch["gen_span_ids"].to(DEVICE)
        gen_mask = batch["gen_span_mask"].to(DEVICE)
        star_y = batch["ordinal_rating"].to(DEVICE)

        outputs = model_p1(input_ids=input_ids, attention_mask=attention_mask, gen_span_ids=gen_ids, gen_span_mask=gen_mask)
        g_loss = outputs["gen_loss"].mean()
        loss = (
            ce_loss_fn(outputs["aspect_logits"].view(-1, cfg.NUM_SENTIMENT_CLASSES), asp_y.view(-1))
            + ce_loss_fn(outputs["aspect_star_logits"].view(-1, cfg.NUM_STARS), asp_star_y.view(-1))
            + ce_loss_fn(outputs["rev_star_logits"], star_y)
            + 0.25 * F.binary_cross_entropy(outputs["rationale_probs"].clamp(1e-6, 1.0 - 1e-6), rat_mask)
            + 0.35 * g_loss
        )
        loss.backward()
        opt_p1.step()
        running_loss += loss.item()

    val_m1 = evaluate_bert_baseline(model_p1, p1_val_loader)
    if val_m1["aspect_macro_f1"] > best_p1_f1:
        best_p1_f1 = val_m1["aspect_macro_f1"]
        raw_state = (model_p1.module if isinstance(model_p1, nn.DataParallel) else model_p1).state_dict()
        torch.save({
            "epoch": epoch,
            "method": "Paper 1: BERT-ABSA + Random Oversampling (Bimaputra & Sutoyo, 2023)",
            "architecture": "paper1_bert",
            "gen_model_name": cfg.GEN_MODEL_NAME,
            "model_state_dict": raw_state,
            "canonical_aspects": cfg.CANONICAL_ASPECTS,
            "val_metrics": val_m1
        }, cfg.MODEL1_SAVE_PATH)
        if cfg.LOCAL_MODEL1_SAVE_PATH.resolve() != cfg.MODEL1_SAVE_PATH.resolve():
            shutil.copy2(cfg.MODEL1_SAVE_PATH, cfg.LOCAL_MODEL1_SAVE_PATH)
    p1_history.append({"epoch": epoch, "train_loss": round(running_loss / max(len(p1_train_loader), 1), 4), **val_m1})
    print(f"[Paper 1] Epoch {epoch}/{cfg.P1_EPOCHS} ({time.time()-t0:.1f}s) | Val Acc@0: {val_m1['acc_at_0']:.2f}% | 6-Aspect F1: {val_m1['aspect_macro_f1']:.2f}% | Macro-MAE: {val_m1['macro_mae']:.4f} | Rat-F1: {val_m1['rationale_f1']:.2f}%")

all_methods_summary["paper1_bert"] = p1_history[-1]
del model_p1, base_p1, opt_p1, p1_train_loader
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 3. Phương pháp 2 — Paper 4: Word2Vec KoHA + BM25 + TF-IGM Linear SVM & CNN (Namee et al., 2023) -> modelmethod4.pt


In [ ]:
try:
    from nltk.stem import WordNetLemmatizer
    lemmatizer = WordNetLemmatizer()
    def lemmatize_tok(w: str) -> str:
        return lemmatizer.lemmatize(w)
except Exception:
    def lemmatize_tok(w: str) -> str:
        return w

SEED_KOHA_KEYWORDS = {
    "Service": ["staff", "service", "friendly", "helpful", "quick", "efficient", "hospitality", "reception", "polite", "attentive", "manager", "concierge", "checkin", "checkout", "waiter"],
    "Facility": ["room", "facility", "facilities", "maintenance", "traffic", "clean", "cleanliness", "comfortable", "bed", "bathroom", "shower", "spacious", "modern", "quiet", "ac", "view"],
    "Experience": ["stay", "experience", "lovely", "perfect", "vacation", "holiday", "relaxing", "atmosphere", "overall", "pleasant", "memorable", "night", "trip", "enjoy", "peaceful"],
    "Amenity": ["kitchen", "food", "taste", "tasty", "breakfast", "pool", "buffet", "spa", "wifi", "restaurant", "bar", "gym", "coffee", "shuttle", "lounge", "massage"],
    "Loyalty": ["return", "back", "again", "times", "visit", "recommend", "recommended", "definitely", "repeat", "future", "choice", "staying", "next", "choose"],
    "Branding": ["price", "value", "expensive", "inexpensive", "luxury", "star", "resort", "brand", "standard", "worth", "prestigious", "class", "reputation"]
}

def paper4_preprocess_tokens(text: str) -> List[str]:
    norm = unicodedata.normalize("NFC", str(text)).lower()
    unigrams = [lemmatize_tok(t) for t in re.findall(r"[a-z]+", norm) if t not in NLTK_STOPWORDS and len(t) > 1]
    bigrams = [f"{unigrams[i]}_{unigrams[i+1]}" for i in range(len(unigrams) - 1)]
    return unigrams + bigrams

print("=== Bắt đầu Huấn luyện Phương pháp 2 (Paper 4: Word2Vec KoHA + BM25 + TF-IGM SVM & CNN) ===")
train_sentences_tokens = []
for txt in df_train_raw["review_text"]:
    for s in re.split(r"[.!?]+", str(txt)):
        toks = paper4_preprocess_tokens(s)
        if toks:
            train_sentences_tokens.append(toks)

word_freq = Counter(t for s in train_sentences_tokens for t in s)
vocab_list = [w for w, c in word_freq.most_common(cfg.P4_MAX_VOCAB) if c >= cfg.P4_W2V_MIN_COUNT]
word2idx = {w: i + 1 for i, w in enumerate(vocab_list)}
idx2word = {i + 1: w for i, w in enumerate(vocab_list)}
vocab_size = len(vocab_list) + 1

cooc_mat = np.zeros((vocab_size, cfg.P4_W2V_DIM), dtype=np.float32)
rng = np.random.default_rng(SEED)
random_proj = rng.normal(0, 1.0 / math.sqrt(cfg.P4_W2V_DIM), size=(vocab_size, cfg.P4_W2V_DIM)).astype(np.float32)

for toks in train_sentences_tokens:
    ids = [word2idx[t] for t in toks if t in word2idx]
    for i, center_id in enumerate(ids):
        left = max(0, i - cfg.P4_W2V_WINDOW)
        right = min(len(ids), i + cfg.P4_W2V_WINDOW + 1)
        for j in range(left, right):
            if i != j:
                cooc_mat[center_id] += random_proj[ids[j]]

norms = np.linalg.norm(cooc_mat, axis=1, keepdims=True)
w2v_embeddings = np.divide(cooc_mat, np.maximum(norms, 1e-6))

koha_Expanded = {}
for asp, seeds in SEED_KOHA_KEYWORDS.items():
    seed_ids = [word2idx[s] for s in seeds if s in word2idx]
    if seed_ids:
        proto = w2v_embeddings[seed_ids].mean(axis=0)
        proto = proto / max(np.linalg.norm(proto), 1e-6)
        sims = w2v_embeddings @ proto
        top_ids = np.argsort(sims)[-120:][::-1]
        koha_Expanded[asp] = set(seeds) | {idx2word[i] for i in top_ids if i in idx2word and sims[i] > 0.25}
    else:
        koha_Expanded[asp] = set(seeds)

N_sents = max(len(train_sentences_tokens), 1)
avg_sent_len = float(np.mean([len(s) for s in train_sentences_tokens])) if train_sentences_tokens else 12.0
doc_freq_sent = Counter()
for s in train_sentences_tokens:
    for tok in set(s):
        doc_freq_sent[tok] += 1

def compute_bm25_score(query_terms: set, sent_tokens: List[str]) -> float:
    if not sent_tokens or not query_terms:
        return 0.0
    s_len = len(sent_tokens)
    tf_map = Counter(sent_tokens)
    score = 0.0
    for q in query_terms:
        if q in tf_map:
            f_qs = tf_map[q]
            df_q = doc_freq_sent.get(q, 0)
            idf_q = math.log((N_sents - df_q + 0.5) / (df_q + 0.5) + 1.0)
            num = f_qs * (cfg.P4_BM25_K1 + 1.0)
            den = f_qs + cfg.P4_BM25_K1 * (1.0 - cfg.P4_BM25_B + cfg.P4_BM25_B * (s_len / max(avg_sent_len, 1.0)))
            score += idf_q * (num / max(den, 1e-6))
    return score

def assemble_aspect_sentences_bm25(review_text: str) -> Dict[str, List[str]]:
    raw_sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", str(review_text)) if s.strip()]
    if not raw_sents and str(review_text).strip():
        raw_sents = [str(review_text).strip()]
    clusters = {asp: [] for asp in cfg.CANONICAL_ASPECTS}
    for s in raw_sents:
        s_toks = paper4_preprocess_tokens(s)
        for asp in cfg.CANONICAL_ASPECTS:
            sc = compute_bm25_score(koha_Expanded[asp], s_toks)
            if sc > 0.0:
                clusters[asp].append((sc, s))
    return {asp: [s for _, s in sorted(clusters[asp], key=lambda x: x[0], reverse=True)] for asp in cfg.CANONICAL_ASPECTS}

class_term_df = np.zeros((vocab_size, cfg.NUM_SENTIMENT_CLASSES), dtype=np.float32)
for _, row in df_train_raw.iterrows():
    for ann in parse_annotations_safe(row.get("aspect_annotations", [])):
        sent_cls = SENTIMENT_TO_ID.get(ann.get("sentiment", "Neutral"), 2)
        for t in set(paper4_preprocess_tokens(ann.get("span_text", ""))):
            if t in word2idx:
                class_term_df[word2idx[t], sent_cls] += 1.0

igm_weights = np.zeros(vocab_size, dtype=np.float32)
for v_i in range(1, vocab_size):
    sorted_f = np.sort(class_term_df[v_i])[::-1]
    denom = sum(sorted_f[r] * (r + 1) for r in range(len(sorted_f)))
    igm_weights[v_i] = float(sorted_f[0] / denom) if denom > 0 else 0.0

def build_sparse_tfigm_matrix(texts: List[str]) -> sp.csr_matrix:
    rows, cols, vals = [], [], []
    for r_i, text_str in enumerate(texts):
        toks = paper4_preprocess_tokens(text_str)
        if not toks:
            continue
        counts = Counter(toks)
        r_cols, r_vals = [], []
        for t, freq in counts.items():
            if t in word2idx:
                vid = word2idx[t]
                w_val = math.log(1.0 + freq) * (1.0 + cfg.P4_TFIGM_LAMBDA * igm_weights[vid])
                r_cols.append(vid)
                r_vals.append(w_val)
        if r_vals:
            norm = math.sqrt(sum(v * v for v in r_vals))
            for c_i, v_i in zip(r_cols, r_vals):
                rows.append(r_i)
                cols.append(c_i)
                vals.append(v_i / max(norm, 1e-6))
    return sp.csr_matrix((vals, (rows, cols)), shape=(len(texts), vocab_size), dtype=np.float32)

class Paper4MultiRegionCNN(nn.Module):
    def __init__(self, cfg: UnifiedBenchmarkConfig, pretrained_w2v: np.ndarray, svm_weights: np.ndarray, svm_biases: np.ndarray, star_svm_w: np.ndarray, star_svm_b: np.ndarray):
        super().__init__()
        self.num_aspects = cfg.NUM_ASPECTS
        self.max_comment_len = cfg.MAX_COMMENT_LEN
        self.num_prompt_tokens = cfg.NUM_SOFT_PROMPT_TOKENS

        self.embedding = nn.Embedding.from_pretrained(torch.tensor(pretrained_w2v, dtype=torch.float32), freeze=False, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=cfg.P4_W2V_DIM, out_channels=cfg.P4_CNN_FILTERS, kernel_size=k)
            for k in cfg.P4_CNN_REGIONS
        ])
        total_filters = len(cfg.P4_CNN_REGIONS) * cfg.P4_CNN_FILTERS
        self.aspect_fc = nn.ModuleList([nn.Linear(total_filters, cfg.NUM_SENTIMENT_CLASSES) for _ in range(cfg.NUM_ASPECTS)])
        self.aspect_star_fc = nn.ModuleList([nn.Linear(total_filters, cfg.NUM_STARS) for _ in range(cfg.NUM_ASPECTS)])
        self.star_fc = nn.Linear(total_filters, cfg.NUM_STARS)

        self.gen_model = AutoModelForSeq2SeqLM.from_pretrained(cfg.GEN_MODEL_NAME)
        self.gen_d_model = self.gen_model.config.d_model
        for param in self.gen_model.parameters():
            param.requires_grad = False

        self.aspect_soft_prompt = nn.ModuleList([
            nn.Sequential(
                nn.Linear(total_filters, 128),
                nn.GELU(),
                nn.Linear(128, self.num_prompt_tokens * self.gen_d_model),
                nn.LayerNorm(self.num_prompt_tokens * self.gen_d_model)
            )
            for _ in range(cfg.NUM_ASPECTS)
        ])

        self.register_buffer("svm_weights", torch.tensor(svm_weights, dtype=torch.float32))
        self.register_buffer("svm_biases", torch.tensor(svm_biases, dtype=torch.float32))
        self.register_buffer("star_svm_w", torch.tensor(star_svm_w, dtype=torch.float32))
        self.register_buffer("star_svm_b", torch.tensor(star_svm_b, dtype=torch.float32))

    def forward_cnn(self, token_ids: torch.Tensor):
        bsz = token_ids.size(0)
        emb = self.embedding(token_ids).transpose(1, 2)
        pooled = [F.max_pool1d(F.relu(conv(emb)), kernel_size=conv(emb).size(-1)).squeeze(-1) for conv in self.convs]
        feat = torch.cat(pooled, dim=-1)
        asp_logits = torch.stack([fc(feat) for fc in self.aspect_fc], dim=1)
        asp_star_logits = torch.stack([fc(feat) for fc in self.aspect_star_fc], dim=1)
        star_logits = self.star_fc(feat)
        soft_prompts = torch.stack([
            sp_head(feat).view(bsz, self.num_prompt_tokens, self.gen_d_model)
            for sp_head in self.aspect_soft_prompt
        ], dim=1)
        return asp_logits, asp_star_logits, star_logits, soft_prompts

    @torch.no_grad()
    def generate_aspect_commentary(self, aspect_idx, aspect_name, sentiment_label, star_val, focused_evidence, soft_prompts, seq2seq_tok):
        device = soft_prompts.device
        prompt_str = (
            f"Paraphrase and summarize the guest's opinion on the hotel {aspect_name} "
            f"({sentiment_label}, {star_val}/5 stars) into a short third-person analytical comment: {focused_evidence}"
        )
        p_enc = seq2seq_tok(prompt_str, max_length=192, truncation=True, return_tensors="pt").to(device)
        tok_embeds = self.gen_model.get_input_embeddings()(p_enc["input_ids"]).float()
        soft_p = soft_prompts[0:1, aspect_idx, :, :].float()
        combined_embeds = torch.cat([0.20 * soft_p, tok_embeds], dim=1)
        prompt_mask = torch.ones((1, soft_p.size(1)), dtype=p_enc["attention_mask"].dtype, device=device)
        combined_mask = torch.cat([prompt_mask, p_enc["attention_mask"]], dim=1)

        enc_out = self.gen_model.encoder(inputs_embeds=combined_embeds, attention_mask=combined_mask)
        gen_ids = self.gen_model.generate(
            encoder_outputs=enc_out,
            attention_mask=combined_mask,
            max_new_tokens=self.max_comment_len,
            min_length=8,
            num_beams=4,
            no_repeat_ngram_size=3,
            repetition_penalty=1.25,
            early_stopping=True
        )
        return seq2seq_tok.decode(gen_ids[0], skip_special_tokens=True).strip()

svm_weights_arr = np.zeros((cfg.NUM_ASPECTS, cfg.NUM_SENTIMENT_CLASSES, vocab_size), dtype=np.float32)
svm_biases_arr = np.zeros((cfg.NUM_ASPECTS, cfg.NUM_SENTIMENT_CLASSES), dtype=np.float32)

X_train_sparse = build_sparse_tfigm_matrix(df_train_raw["review_text"].tolist())
y_train_star = df_train_raw["ordinal_rating"].values.astype(int)

star_svm = LinearSVC(C=1.0, random_state=SEED)
star_svm.fit(X_train_sparse, y_train_star)
star_svm_w = np.zeros((cfg.NUM_STARS, vocab_size), dtype=np.float32)
star_svm_b = np.zeros(cfg.NUM_STARS, dtype=np.float32)
for idx_c, cls_v in enumerate(star_svm.classes_):
    if idx_c < star_svm.coef_.shape[0]:
        star_svm_w[int(cls_v)] = star_svm.coef_[idx_c].astype(np.float32)
        star_svm_b[int(cls_v)] = star_svm.intercept_[idx_c].astype(np.float32)

Y_train_aspects = np.zeros((len(df_train_raw), cfg.NUM_ASPECTS), dtype=int)
Y_train_asp_stars = np.zeros((len(df_train_raw), cfg.NUM_ASPECTS), dtype=int)
for idx_r, row in df_train_raw.iterrows():
    ord_r = int(row["ordinal_rating"])
    Y_train_asp_stars[idx_r, :] = ord_r
    votes = defaultdict(list)
    spans_map = defaultdict(list)
    for ann in parse_annotations_safe(row.get("aspect_annotations", [])):
        asp = ann.get("aspect")
        if asp in ASPECT_ID_MAP:
            a_i = ASPECT_ID_MAP[asp] - 1
            votes[a_i].append(SENTIMENT_TO_ID.get(ann.get("sentiment", "Neutral"), 2))
            spans_map[a_i].append(str(ann.get("span_text", "")))
    for a_i, v_l in votes.items():
        maj = Counter(v_l).most_common(1)[0][0]
        Y_train_aspects[idx_r, a_i] = maj
        sp_words = re.findall(r"[a-z]+", " ".join(spans_map[a_i]).lower())
        has_hedge = any(w in HEDGING_MODIFIERS for w in sp_words)
        if maj == 3:
            Y_train_asp_stars[idx_r, a_i] = 3 if has_hedge else max(3, ord_r)
        elif maj == 2:
            Y_train_asp_stars[idx_r, a_i] = 2
        else:
            Y_train_asp_stars[idx_r, a_i] = 1 if has_hedge else min(1, ord_r)

for a_i, asp in enumerate(cfg.CANONICAL_ASPECTS):
    clf_svm = LinearSVC(C=1.0, random_state=SEED)
    clf_svm.fit(X_train_sparse, Y_train_aspects[:, a_i])
    for idx_c, cls_v in enumerate(clf_svm.classes_):
        if idx_c < clf_svm.coef_.shape[0]:
            svm_weights_arr[a_i, int(cls_v)] = clf_svm.coef_[idx_c].astype(np.float32)
            svm_biases_arr[a_i, int(cls_v)] = clf_svm.intercept_[idx_c].astype(np.float32)

def encode_seq_ids(text_str: str, max_len: int = 128) -> List[int]:
    toks = paper4_preprocess_tokens(text_str)
    ids = [word2idx[t] for t in toks if t in word2idx][:max_len]
    if len(ids) < max_len:
        ids = ids + [0] * (max_len - len(ids))
    return ids

X_seq_train = torch.tensor([encode_seq_ids(t, cfg.P4_MAX_LEN) for t in df_train_raw["review_text"]], dtype=torch.long)
Y_asp_train_t = torch.tensor(Y_train_aspects, dtype=torch.long)
Y_asp_star_train_t = torch.tensor(Y_train_asp_stars, dtype=torch.long)
Y_star_train_t = torch.tensor(y_train_star, dtype=torch.long)

model_p4 = Paper4MultiRegionCNN(cfg, w2v_embeddings, svm_weights_arr, svm_biases_arr, star_svm_w, star_svm_b).to(DEVICE)
with torch.no_grad():
    aspect_proto_targets = []
    for asp in cfg.CANONICAL_ASPECTS:
        kw_str = f"{asp}: " + " ".join(SEED_KOHA_KEYWORDS[asp])
        p_enc = gen_tokenizer(kw_str, return_tensors="pt", truncation=True, max_length=48).to(DEVICE)
        t5_h = model_p4.gen_model.get_input_embeddings()(p_enc["input_ids"]).float()
        aspect_proto_targets.append(t5_h.mean(dim=1).squeeze(0))
    aspect_proto_targets = torch.stack(aspect_proto_targets, dim=0)

cnn_loader = DataLoader(list(zip(X_seq_train, Y_asp_train_t, Y_asp_star_train_t, Y_star_train_t)), batch_size=cfg.P4_BATCH_SIZE, shuffle=True)
opt_p4 = torch.optim.Adam([p for p in model_p4.parameters() if p.requires_grad], lr=cfg.P4_LR)

for epoch in range(1, cfg.P4_EPOCHS + 1):
    model_p4.train()
    for bx, by_asp, by_asp_star, by_star in cnn_loader:
        bx, by_asp, by_asp_star, by_star = bx.to(DEVICE), by_asp.to(DEVICE), by_asp_star.to(DEVICE), by_star.to(DEVICE)
        opt_p4.zero_grad()
        asp_l, asp_star_l, star_l, soft_p = model_p4.forward_cnn(bx)
        pred_sp_mean = soft_p.float().mean(dim=2)
        target_sp = aspect_proto_targets.unsqueeze(0).expand(bx.size(0), -1, -1)
        sp_loss = (1.0 - F.cosine_similarity(pred_sp_mean, target_sp, dim=-1).mean()) + F.smooth_l1_loss(pred_sp_mean, target_sp)
        loss = (
            ce_loss_fn(asp_l.view(-1, cfg.NUM_SENTIMENT_CLASSES), by_asp.view(-1))
            + ce_loss_fn(asp_star_l.view(-1, cfg.NUM_STARS), by_asp_star.view(-1))
            + ce_loss_fn(star_l, by_star)
            + 0.35 * sp_loss
        )
        loss.backward()
        opt_p4.step()

def evaluate_paper4(df_eval: pd.DataFrame) -> Dict[str, Any]:
    model_p4.eval()
    X_ev_sparse = build_sparse_tfigm_matrix(df_eval["review_text"].tolist())
    X_seq_all = torch.tensor([encode_seq_ids(t, cfg.P4_MAX_LEN) for t in df_eval["review_text"]], dtype=torch.long)

    y_pred_star_list = []
    Y_pred_asp_list = []
    svm_w_np = model_p4.svm_weights.detach().cpu().numpy()
    svm_b_np = model_p4.svm_biases.detach().cpu().numpy()
    star_w_np = model_p4.star_svm_w.detach().cpu().numpy()
    star_b_np = model_p4.star_svm_b.detach().cpu().numpy()

    with torch.no_grad():
        for start_i in range(0, len(df_eval), cfg.P4_BATCH_SIZE):
            end_i = min(len(df_eval), start_i + cfg.P4_BATCH_SIZE)
            bx_seq = X_seq_all[start_i:end_i].to(DEVICE)
            cnn_asp_l, _, cnn_star_l, _ = model_p4.forward_cnn(bx_seq)
            cnn_asp_np = cnn_asp_l.cpu().numpy()
            cnn_star_np = cnn_star_l.cpu().numpy()

            bx_sparse = X_ev_sparse[start_i:end_i]
            svm_star_np = bx_sparse.dot(star_w_np.T) + star_b_np
            svm_asp_np = np.stack([bx_sparse.dot(svm_w_np[a].T) + svm_b_np[a] for a in range(cfg.NUM_ASPECTS)], axis=1)

            blended_star = 0.7 * svm_star_np + 0.3 * cnn_star_np
            blended_asp = 0.7 * svm_asp_np + 0.3 * cnn_asp_np

            y_pred_star_list.append(blended_star.argmax(axis=-1))
            Y_pred_asp_list.append(blended_asp.argmax(axis=-1))

    y_pred_star = np.concatenate(y_pred_star_list, axis=0)
    Y_pred_asp = np.vstack(Y_pred_asp_list)
    y_true_star = df_eval["ordinal_rating"].values.astype(int)
    Y_true_asp = np.zeros((len(df_eval), cfg.NUM_ASPECTS), dtype=int)

    rat_inter, rat_pred_sum, rat_true_sum = 0.0, 0.0, 0.0
    overlap_rates = []
    for idx_r, row in df_eval.iterrows():
        true_toks_per_asp = defaultdict(set)
        for ann in parse_annotations_safe(row.get("aspect_annotations", [])):
            asp = ann.get("aspect")
            if asp in ASPECT_ID_MAP:
                a_i = ASPECT_ID_MAP[asp] - 1
                Y_true_asp[idx_r, a_i] = SENTIMENT_TO_ID.get(ann.get("sentiment", "Neutral"), 2)
                true_toks_per_asp[a_i].update(paper4_preprocess_tokens(ann.get("span_text", "")))

        bm25_clusters = assemble_aspect_sentences_bm25(row["review_text"])
        tok_aspect_count = Counter()
        for a_i, asp in enumerate(cfg.CANONICAL_ASPECTS):
            sents = bm25_clusters.get(asp, [])
            ptoks = set(paper4_preprocess_tokens(" ".join(sents[:1]))) if sents else set()
            for t in ptoks:
                tok_aspect_count[t] += 1
            t_set = true_toks_per_asp[a_i]
            rat_inter += len(ptoks & t_set)
            rat_pred_sum += len(ptoks)
            rat_true_sum += len(t_set)

        if tok_aspect_count:
            leaked = sum(1 for c in tok_aspect_count.values() if c > 1)
            overlap_rates.append(leaked / len(tok_aspect_count))

    acc0 = accuracy_score(y_true_star, y_pred_star) * 100.0
    per_star_mae = [np.abs(y_true_star[y_true_star == k] - y_pred_star[y_true_star == k]).mean() for k in range(5) if np.any(y_true_star == k)]
    macro_mae = float(np.mean(per_star_mae)) if per_star_mae else 0.0
    asp_f1s = [f1_score(Y_true_asp[:, a], Y_pred_asp[:, a], average="macro", zero_division=0) * 100.0 for a in range(cfg.NUM_ASPECTS)]
    rationale_f1 = (2.0 * rat_inter / max(rat_pred_sum + rat_true_sum, 1.0)) * 100.0
    leakage_rate = float(np.mean(overlap_rates)) * 100.0 if overlap_rates else 0.0

    return {
        "val_loss": 0.0,
        "acc_at_0": round(acc0, 2),
        "aspect_macro_f1": round(float(np.mean(asp_f1s)), 2),
        "macro_mae": round(macro_mae, 4),
        "rationale_f1": round(rationale_f1, 2),
        "leakage_rate": round(leakage_rate, 2),
        "per_aspect_f1": dict(zip(cfg.CANONICAL_ASPECTS, [round(x, 2) for x in asp_f1s]))
    }

val_m4 = evaluate_paper4(df_val)
print(f"[Paper 4] Val Acc@0: {val_m4['acc_at_0']:.2f}% | 6-Aspect F1: {val_m4['aspect_macro_f1']:.2f}% | Macro-MAE: {val_m4['macro_mae']:.4f} | Rat-F1: {val_m4['rationale_f1']:.2f}%")
torch.save({
    "epoch": cfg.P4_EPOCHS,
    "method": "Paper 4: Word2Vec KoHA + BM25 + TF-IGM SVM & CNN (Namee et al., 2023)",
    "architecture": "paper4_bm25_tfigm_svm_cnn",
    "gen_model_name": cfg.GEN_MODEL_NAME,
    "model_state_dict": model_p4.state_dict(),
    "w2v_embeddings": w2v_embeddings,
    "word2idx": word2idx,
    "igm_weights": igm_weights,
    "N_sents": N_sents,
    "avg_sent_len": avg_sent_len,
    "doc_freq_sent": dict(doc_freq_sent),
    "koha_expanded": {k: list(v) for k, v in koha_Expanded.items()},
    "canonical_aspects": cfg.CANONICAL_ASPECTS,
    "val_metrics": val_m4
}, cfg.MODEL4_SAVE_PATH)
if cfg.LOCAL_MODEL4_SAVE_PATH.resolve() != cfg.MODEL4_SAVE_PATH.resolve():
    shutil.copy2(cfg.MODEL4_SAVE_PATH, cfg.LOCAL_MODEL4_SAVE_PATH)

all_methods_summary["paper4_bm25_tfigm_svm_cnn"] = val_m4
del model_p4, opt_p4, cnn_loader, X_seq_train
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 4. Phương pháp 3 — Paper 9: WSD-BERT + BiLSTM + Attention + 2-Layer GCN (Nadeem et al., 2025) -> modelmethod9.pt


In [ ]:
try:
    from nltk.corpus import wordnet as wn
    WORDNET_AVAILABLE = True
except Exception:
    WORDNET_AVAILABLE = False

AMBIGUOUS_HOTEL_TERMS = {
    "sharp", "cold", "warm", "hard", "soft", "small", "high", "low",
    "light", "heavy", "old", "dark", "basic", "quiet", "plain", "rich", "fair", "fine"
}

def enrich_text_with_wordnet_wsd(text: str) -> str:
    raw = unicodedata.normalize("NFC", str(text).strip())
    if not WORDNET_AVAILABLE:
        return raw
    words = re.findall(r"[a-zA-Z]+", raw)
    context_set = {w.lower() for w in words if len(w) > 2}
    sense_glosses = []
    for w in words:
        wl = w.lower()
        if wl in AMBIGUOUS_HOTEL_TERMS:
            synsets = wn.synsets(wl)
            if synsets:
                best_syn = synsets[0]
                best_sim = -1.0
                for syn in synsets[:4]:
                    gloss_words = set(re.findall(r"[a-z]+", (syn.definition() + " " + " ".join(syn.examples())).lower()))
                    if gloss_words:
                        inter = len(context_set & gloss_words)
                        cos_sim = inter / math.sqrt(max(len(context_set), 1) * len(gloss_words))
                        if cos_sim > best_sim:
                            best_sim = cos_sim
                            best_syn = syn
                sense_glosses.append(f"{wl}: {best_syn.definition()}")
                if len(sense_glosses) >= 2:
                    break
    if sense_glosses:
        return raw + " [SEP] " + " ; ".join(sense_glosses)
    return raw

def build_normalized_dependency_adj(attention_mask: torch.Tensor, window: int = 2) -> torch.Tensor:
    bsz, seq_len = attention_mask.size()
    idx = torch.arange(seq_len, device=attention_mask.device)
    dist = (idx.unsqueeze(0) - idx.unsqueeze(1)).abs()
    local_adj = (dist <= window).float().unsqueeze(0).expand(bsz, -1, -1)
    valid_2d = attention_mask.unsqueeze(1).float() * attention_mask.unsqueeze(2).float()
    adj = local_adj * valid_2d
    adj[:, 0, :] = attention_mask.float()
    adj[:, :, 0] = attention_mask.float()
    deg = adj.sum(dim=-1, keepdim=True).clamp(min=1.0)
    inv_sqrt_deg = deg.pow(-0.5)
    return inv_sqrt_deg * adj * inv_sqrt_deg.transpose(1, 2)

class Paper9WSDDataset(Dataset):
    def __init__(self, df: pd.DataFrame, bert_tok, seq2seq_tok, max_len: int = 256, max_comment_len: int = 48):
        self.df = df.reset_index(drop=True)
        self.bert_tok = bert_tok
        self.seq2seq_tok = seq2seq_tok
        self.max_len = max_len
        self.max_comment_len = max_comment_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        raw_text = unicodedata.normalize("NFC", str(row["review_text"]))
        wsd_text = enrich_text_with_wordnet_wsd(raw_text)
        enc = self.bert_tok(
            wsd_text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_offsets_mapping=True,
            return_tensors="pt"
        )
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        offsets = enc["offset_mapping"].squeeze(0).numpy()

        ordinal_rating = int(row["ordinal_rating"])
        aspect_sent_labels = torch.zeros(cfg.NUM_ASPECTS, dtype=torch.long)
        aspect_star_labels = torch.full((cfg.NUM_ASPECTS,), ordinal_rating, dtype=torch.long)
        rationale_masks = torch.zeros((cfg.NUM_ASPECTS, self.max_len), dtype=torch.float32)
        gen_span_ids = torch.zeros((cfg.NUM_ASPECTS, self.max_comment_len), dtype=torch.long)
        gen_span_mask = torch.zeros((cfg.NUM_ASPECTS, self.max_comment_len), dtype=torch.long)

        votes = defaultdict(list)
        span_texts_map = defaultdict(list)
        for ann in parse_annotations_safe(row.get("aspect_annotations", [])):
            asp = ann.get("aspect")
            if asp in ASPECT_ID_MAP:
                a_idx = ASPECT_ID_MAP[asp] - 1
                votes[a_idx].append(SENTIMENT_TO_ID.get(ann.get("sentiment", "Neutral"), 2))
                raw_sp = str(ann.get("span_text", "")).strip()
                if raw_sp:
                    span_texts_map[a_idx].append(raw_sp)
                s_char = int(ann.get("start_char", 0))
                e_char = int(ann.get("end_char", 0))
                for t_i, (ts, te) in enumerate(offsets):
                    if te > ts and ts < e_char and te > s_char:
                        rationale_masks[a_idx, t_i] = 1.0

        for a_idx, v_list in votes.items():
            maj = Counter(v_list).most_common(1)[0][0]
            aspect_sent_labels[a_idx] = maj
            joined_sp = " ".join(span_texts_map[a_idx]).strip()
            sp_words = re.findall(r"[a-z]+", (joined_sp if joined_sp else raw_text).lower())
            has_hedge = any(w in HEDGING_MODIFIERS for w in sp_words)
            if maj == 3:
                aspect_star_labels[a_idx] = 3 if has_hedge else max(3, ordinal_rating)
            elif maj == 2:
                aspect_star_labels[a_idx] = 2
            else:
                aspect_star_labels[a_idx] = 1 if has_hedge else min(1, ordinal_rating)

            if joined_sp:
                t_enc = self.seq2seq_tok(
                    joined_sp,
                    max_length=self.max_comment_len,
                    padding="max_length",
                    truncation=True,
                    return_tensors="pt"
                )
                gen_span_ids[a_idx] = t_enc["input_ids"].squeeze(0)
                gen_span_mask[a_idx] = t_enc["attention_mask"].squeeze(0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "aspect_sent_labels": aspect_sent_labels,
            "aspect_star_labels": aspect_star_labels,
            "rationale_masks": rationale_masks,
            "gen_span_ids": gen_span_ids,
            "gen_span_mask": gen_span_mask,
            "ordinal_rating": torch.tensor(ordinal_rating, dtype=torch.long)
        }

class GraphConvLayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, dropout: float = 0.5):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, norm_adj: torch.Tensor) -> torch.Tensor:
        return self.dropout(F.relu(torch.bmm(norm_adj, self.linear(x))))

class Paper9WSD_BiLSTM_GCN(nn.Module):
    def __init__(self, cfg: UnifiedBenchmarkConfig):
        super().__init__()
        self.num_aspects = cfg.NUM_ASPECTS
        self.max_comment_len = cfg.MAX_COMMENT_LEN
        self.num_prompt_tokens = cfg.NUM_SOFT_PROMPT_TOKENS

        self.bert = AutoModel.from_pretrained(cfg.BACKBONE_NAME)
        self.gen_model = AutoModelForSeq2SeqLM.from_pretrained(cfg.GEN_MODEL_NAME)
        self.gen_d_model = self.gen_model.config.d_model
        for param in self.gen_model.parameters():
            param.requires_grad = False

        self.bilstm = nn.LSTM(
            input_size=cfg.P9_BERT_DIM,
            hidden_size=cfg.P9_BILSTM_UNITS,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        self.bilstm_dropout = nn.Dropout(cfg.P9_DROPOUT)
        self.attn_queries = nn.Parameter(torch.randn(cfg.NUM_ASPECTS, cfg.P9_BILSTM_UNITS * 2) * 0.02)

        self.gcn1 = GraphConvLayer(cfg.P9_BERT_DIM, cfg.P9_GCN_DIM, cfg.P9_DROPOUT)
        self.gcn2 = GraphConvLayer(cfg.P9_GCN_DIM, cfg.P9_GCN_DIM, cfg.P9_DROPOUT)

        combined_dim = cfg.P9_BILSTM_UNITS * 2 + cfg.P9_GCN_DIM
        self.fusion_gate = nn.Parameter(torch.tensor(0.5))

        self.aspect_classifiers = nn.ModuleList([
            nn.Sequential(nn.Dropout(cfg.P9_DROPOUT), nn.Linear(combined_dim, cfg.NUM_SENTIMENT_CLASSES))
            for _ in range(cfg.NUM_ASPECTS)
        ])
        self.aspect_star_heads = nn.ModuleList([
            nn.Sequential(nn.Dropout(cfg.P9_DROPOUT), nn.Linear(combined_dim, cfg.NUM_STARS))
            for _ in range(cfg.NUM_ASPECTS)
        ])
        self.review_star_head = nn.Sequential(nn.Dropout(cfg.P9_DROPOUT), nn.Linear(combined_dim, cfg.NUM_STARS))
        self.aspect_soft_prompt = nn.Sequential(
            nn.Linear(combined_dim, 256),
            nn.GELU(),
            nn.Linear(256, self.num_prompt_tokens * self.gen_d_model),
            nn.LayerNorm(self.num_prompt_tokens * self.gen_d_model)
        )

    def forward(self, input_ids, attention_mask, gen_span_ids=None, gen_span_mask=None):
        bsz = input_ids.size(0)
        self.bilstm.flatten_parameters()
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        seq_emb = bert_out.last_hidden_state

        lstm_out, _ = self.bilstm(seq_emb)
        lstm_out = self.bilstm_dropout(lstm_out)

        norm_adj = build_normalized_dependency_adj(attention_mask, window=2)
        gcn_h = self.gcn2(self.gcn1(seq_emb, norm_adj), norm_adj)

        pad_mask = (attention_mask == 0).unsqueeze(1)
        attn_scores = torch.einsum("bld,ad->bal", lstm_out, self.attn_queries) / math.sqrt(lstm_out.size(-1))
        attn_scores = attn_scores.masked_fill(pad_mask, -1e4)
        attn_weights = F.softmax(attn_scores, dim=-1)
        rationale_probs = torch.sigmoid(attn_scores) * attention_mask.unsqueeze(1).float()

        lstm_Side = torch.einsum("bal,bld->bad", attn_weights, lstm_out)
        gcn_Side = torch.einsum("bal,bld->bad", attn_weights, gcn_h)

        w = torch.sigmoid(self.fusion_gate)
        combined = torch.cat([w * lstm_Side, (1.0 - w) * gcn_Side], dim=-1)

        asp_logits = torch.stack([self.aspect_classifiers[a](combined[:, a, :]) for a in range(self.num_aspects)], dim=1)
        asp_star_logits = torch.stack([self.aspect_star_heads[a](combined[:, a, :]) for a in range(self.num_aspects)], dim=1)
        rev_star_logits = self.review_star_head(combined.mean(dim=1))
        soft_prompts = torch.stack([
            self.aspect_soft_prompt(combined[:, a, :]).view(bsz, self.num_prompt_tokens, self.gen_d_model)
            for a in range(self.num_aspects)
        ], dim=1)

        gen_loss = torch.zeros(1, device=input_ids.device, dtype=torch.float32)
        if gen_span_ids is not None and gen_span_mask is not None:
            flat_prompts = soft_prompts.view(bsz * self.num_aspects, self.num_prompt_tokens, self.gen_d_model)
            flat_ids = gen_span_ids.view(bsz * self.num_aspects, -1)
            flat_mask = gen_span_mask.view(bsz * self.num_aspects, -1)
            valid_rows = flat_mask.sum(dim=-1) > 1
            if valid_rows.any():
                active_prompts = flat_prompts[valid_rows].float()
                active_ids = flat_ids[valid_rows]
                active_mask = flat_mask[valid_rows]
                with torch.no_grad():
                    t5_tok_emb = self.gen_model.get_input_embeddings()(active_ids).float()
                    mask_f = active_mask.unsqueeze(-1).float()
                    target_vec = (t5_tok_emb * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)
                pred_vec = active_prompts.mean(dim=1)
                gen_loss = ((1.0 - F.cosine_similarity(pred_vec, target_vec, dim=-1).mean()) + F.smooth_l1_loss(pred_vec, target_vec)).view(1)

        return {
            "aspect_logits": asp_logits,
            "aspect_star_logits": asp_star_logits,
            "rev_star_logits": rev_star_logits,
            "rationale_probs": rationale_probs,
            "soft_prompts": soft_prompts,
            "gen_loss": gen_loss
        }

    @torch.no_grad()
    def generate_aspect_commentary(self, aspect_idx, aspect_name, sentiment_label, star_val, review_text, outputs, input_ids, attention_mask, bert_tok, seq2seq_tok):
        device = input_ids.device
        rat_prob = outputs["rationale_probs"][0, aspect_idx]
        valid_len = int(attention_mask[0].sum().item())
        tok_ids = input_ids[0, :valid_len].cpu().tolist()
        scores = rat_prob[:valid_len].cpu().numpy()

        sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", review_text.strip()) if s.strip()]
        if len(sentences) > 1:
            scored_sents = []
            for s in sentences:
                s_ids = set(bert_tok.encode(s, add_special_tokens=False))
                hit_scores = [scores[i] for i in range(1, valid_len - 1) if tok_ids[i] in s_ids]
                scored_sents.append((float(np.mean(hit_scores)) if hit_scores else 0.0, s))
            scored_sents.sort(key=lambda x: x[0], reverse=True)
            focused_evidence = " ".join([s for _, s in scored_sents[:2]])
        else:
            focused_evidence = review_text.strip()

        prompt_str = (
            f"Paraphrase and summarize the guest's opinion on the hotel {aspect_name} "
            f"({sentiment_label}, {star_val}/5 stars) into a short third-person analytical comment: {focused_evidence}"
        )
        p_enc = seq2seq_tok(prompt_str, max_length=192, truncation=True, return_tensors="pt").to(device)
        tok_embeds = self.gen_model.get_input_embeddings()(p_enc["input_ids"]).float()
        soft_p = outputs["soft_prompts"][0:1, aspect_idx, :, :].float()
        combined_embeds = torch.cat([0.20 * soft_p, tok_embeds], dim=1)
        prompt_mask = torch.ones((1, soft_p.size(1)), dtype=p_enc["attention_mask"].dtype, device=device)
        combined_mask = torch.cat([prompt_mask, p_enc["attention_mask"]], dim=1)

        enc_out = self.gen_model.encoder(inputs_embeds=combined_embeds, attention_mask=combined_mask)
        gen_ids = self.gen_model.generate(
            encoder_outputs=enc_out,
            attention_mask=combined_mask,
            max_new_tokens=self.max_comment_len,
            min_length=8,
            num_beams=4,
            no_repeat_ngram_size=3,
            repetition_penalty=1.25,
            early_stopping=True
        )
        return seq2seq_tok.decode(gen_ids[0], skip_special_tokens=True).strip()

print("=== Bắt đầu Huấn luyện Phương pháp 3 (Paper 9: WSD-BERT + BiLSTM + Attention + 2-Layer GCN) ===")
p9_train_loader = DataLoader(Paper9WSDDataset(df_train_raw, bert_tokenizer, gen_tokenizer, cfg.MAX_SEQ_LEN, cfg.MAX_COMMENT_LEN), batch_size=cfg.P9_BATCH_SIZE, shuffle=True, num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY)
p9_val_loader = DataLoader(Paper9WSDDataset(df_val, bert_tokenizer, gen_tokenizer, cfg.MAX_SEQ_LEN, cfg.MAX_COMMENT_LEN), batch_size=cfg.P9_BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY)

base_p9 = Paper9WSD_BiLSTM_GCN(cfg).to(DEVICE)
model_p9 = nn.DataParallel(base_p9, device_ids=list(range(NUM_GPUS))) if NUM_GPUS > 1 else base_p9
raw_p9 = model_p9.module if isinstance(model_p9, nn.DataParallel) else model_p9
opt_p9 = torch.optim.AdamW([
    {"params": [p for p in raw_p9.bert.parameters() if p.requires_grad], "lr": cfg.P9_LR_BERT, "weight_decay": 0.01},
    {"params": [p for n, p in raw_p9.named_parameters() if not n.startswith("bert.") and not n.startswith("gen_model.") and p.requires_grad], "lr": cfg.P9_LR_HEADS, "weight_decay": 0.01}
])

best_p9_f1 = -1.0
p9_history = []
for epoch in range(1, cfg.P9_EPOCHS + 1):
    model_p9.train()
    t0 = time.time()
    running_loss = 0.0
    for batch in p9_train_loader:
        opt_p9.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        asp_y = batch["aspect_sent_labels"].to(DEVICE)
        asp_star_y = batch["aspect_star_labels"].to(DEVICE)
        rat_mask = batch["rationale_masks"].to(DEVICE)
        gen_ids = batch["gen_span_ids"].to(DEVICE)
        gen_mask = batch["gen_span_mask"].to(DEVICE)
        star_y = batch["ordinal_rating"].to(DEVICE)

        outputs = model_p9(input_ids=input_ids, attention_mask=attention_mask, gen_span_ids=gen_ids, gen_span_mask=gen_mask)
        g_loss = outputs["gen_loss"].mean()
        loss = (
            ce_loss_fn(outputs["aspect_logits"].view(-1, cfg.NUM_SENTIMENT_CLASSES), asp_y.view(-1))
            + ce_loss_fn(outputs["aspect_star_logits"].view(-1, cfg.NUM_STARS), asp_star_y.view(-1))
            + ce_loss_fn(outputs["rev_star_logits"], star_y)
            + 0.25 * F.binary_cross_entropy(outputs["rationale_probs"].clamp(1e-6, 1.0 - 1e-6), rat_mask)
            + 0.35 * g_loss
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_p9.parameters(), 1.0)
        opt_p9.step()
        running_loss += loss.item()

    val_m9 = evaluate_bert_baseline(model_p9, p9_val_loader)
    if val_m9["aspect_macro_f1"] > best_p9_f1:
        best_p9_f1 = val_m9["aspect_macro_f1"]
        raw_state = (model_p9.module if isinstance(model_p9, nn.DataParallel) else model_p9).state_dict()
        torch.save({
            "epoch": epoch,
            "method": "Paper 9: WSD-BERT + BiLSTM + Attention + GCN (Nadeem et al., 2025)",
            "architecture": "paper9_wsd_bilstm_gcn",
            "gen_model_name": cfg.GEN_MODEL_NAME,
            "model_state_dict": raw_state,
            "canonical_aspects": cfg.CANONICAL_ASPECTS,
            "val_metrics": val_m9
        }, cfg.MODEL9_SAVE_PATH)
        if cfg.LOCAL_MODEL9_SAVE_PATH.resolve() != cfg.MODEL9_SAVE_PATH.resolve():
            shutil.copy2(cfg.MODEL9_SAVE_PATH, cfg.LOCAL_MODEL9_SAVE_PATH)
    p9_history.append({"epoch": epoch, "train_loss": round(running_loss / max(len(p9_train_loader), 1), 4), **val_m9})
    print(f"[Paper 9] Epoch {epoch}/{cfg.P9_EPOCHS} ({time.time()-t0:.1f}s) | Val Acc@0: {val_m9['acc_at_0']:.2f}% | 6-Aspect F1: {val_m9['aspect_macro_f1']:.2f}% | Macro-MAE: {val_m9['macro_mae']:.4f} | Rat-F1: {val_m9['rationale_f1']:.2f}%")

all_methods_summary["paper9_wsd_bilstm_gcn"] = p9_history[-1]
del model_p9, base_p9, opt_p9, p9_train_loader
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 5. Tổng hợp Kết quả Đánh giá Cả 3 Phương pháp (modelmethod1.pt, modelmethod4.pt, modelmethod9.pt)


In [ ]:
with open(cfg.SUMMARY_SAVE_PATH, "w", encoding="utf-8") as f:
    json.dump(all_methods_summary, f, indent=2, ensure_ascii=False)

summary_rows = []
for m_key, m_val in all_methods_summary.items():
    summary_rows.append({
        "Method": m_key,
        "Acc@0 (%)": m_val["acc_at_0"],
        "6-Aspect Macro-F1 (%)": m_val["aspect_macro_f1"],
        "Macro-MAE": m_val["macro_mae"],
        "Rationale-F1 (%)": m_val["rationale_f1"],
        "Leakage (%)": m_val["leakage_rate"]
    })
df_summary = pd.DataFrame(summary_rows)
print("\n=== BẢNG TỔNG HỢP KẾT QUẢ 3 PHƯƠNG PHÁP ĐỐI CHỨNG ===")
print(df_summary.to_string(index=False))
print(f"\nĐã lưu thành công 3 file trọng số: {cfg.MODEL1_SAVE_PATH}, {cfg.MODEL4_SAVE_PATH}, {cfg.MODEL9_SAVE_PATH}")
